In [ ]:
import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.base import BaseEstimator, ClassifierMixin
from sklearn.naive_bayes import GaussianNB, CategoricalNB, BernoulliNB
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score
from pprint import pprint

In [ ]:
# Define the Raw Data URLs
batter_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Batters-FullSeason.csv"
defense_url = "https://github.com/willcreager/willcreager.github.io/raw/refs/heads/main/docs/data/csvs/MLBData-ABSChallengeablePitches-Defense-FullSeason.csv"

In [ ]:
class HybridNaiveBayes(BaseEstimator, ClassifierMixin):
    def __init__(self, use_balancing=False, bias=0.0):
        # use_balancing: True shifts the decision threshold to handle class imbalance
        # bias: Float to fine-tune the strictness. Higher values increase "Not Challenged" predictions.
        self.use_balancing = use_balancing
        self.bias = bias
        self.gaussian_model = GaussianNB()
        self.categorical_model = CategoricalNB()
        self.bernoulli_model = BernoulliNB()
        self.classes_ = None
        self.class_log_prior_ = None
        self.empirical_class_prior_ = None

    def _split_features(self, X):
        gaussian_cols = ["plateTimeX", "plateTimeY_topDiff", "plateTimeY_BotDiff", "startSpeed", "inning", "netScore"]
        categorical_cols = ["prePitchBalls", "prePitchStrikes", "outs", "challengesLeft", "pitchTypeGroup"]
        bernoulli_cols = ["runnerOn1st", "runnerOn2nd", "runnerOn3rd"]

        gaussian_existing = [col for col in gaussian_cols if col in X.columns]
        categorical_existing = [col for col in categorical_cols if col in X.columns]
        bernoulli_existing = [col for col in bernoulli_cols if col in X.columns]

        return X[gaussian_existing], X[categorical_existing], X[bernoulli_existing]

    def fit(self, X, y):
        self.classes_ = np.unique(y)

        # Safely compute empirical priors from training labels directly
        _, counts = np.unique(y, return_counts=True)
        self.empirical_class_prior_ = counts / len(y)

        X_gauss, X_cat, X_bern = self._split_features(X)

        # Train models naturally on the empirical distribution
        self.gaussian_model.fit(X_gauss, y)
        self.categorical_model.fit(X_cat, y)
        self.bernoulli_model.fit(X_bern, y)

        self.class_log_prior_ = np.log(self.empirical_class_prior_)
        self.class_prior_ = self.empirical_class_prior_
        self.class_count_ = counts
        return self

    def predict_joint_log_proba(self, X):
        X_gauss, X_cat, X_bern = self._split_features(X)

        log_prob_gauss = self.gaussian_model.predict_joint_log_proba(X_gauss)
        log_prob_cat = self.categorical_model.predict_joint_log_proba(X_cat)
        log_prob_bern = self.bernoulli_model.predict_joint_log_proba(X_bern)

        # Sum the joint likelihoods, subtracting out duplicate priors added by the sub-models
        joint_log_likelihood = log_prob_gauss + log_prob_cat + log_prob_bern - 2 * self.class_log_prior_
        return joint_log_likelihood

    def predict(self, X):
        joint_log_proba = self.predict_joint_log_proba(X)

        if self.use_balancing and len(self.classes_) == 2:
            # Calculate the explicit mathematical log-odds penalty for the imbalanced background rate
            log_odds_adjustment = np.log(self.empirical_class_prior_[1] / self.empirical_class_prior_[0])

            # Apply threshold shift to the "Challenged" class (Index 0) vs 'Not Challenged' (Index 1)
            # Higher bias values make the model more conservative about predicting a challenge
            decision_threshold = log_odds_adjustment + self.bias

            log_odds_features = joint_log_proba[:, 0] - joint_log_proba[:, 1]
            return np.where(log_odds_features > -decision_threshold, self.classes_[0], self.classes_[1])

        return self.classes_[np.argmax(joint_log_proba, axis=1)]


# Using NB to predict whether batter will challenge

In [ ]:
# Load batting data
df_batters = pd.read_csv(batter_url, encoding="latin1")

# Remove rows with null pitch locations
df_batters = df_batters.dropna(subset=["plateTimeY", "plateTimeX"])

# Feature Engineering: Calculate vertical distance deviations from zone edges
# Positive values mean above the edge, negative values mean below the edge
df_batters["plateTimeY_topDiff"] = (
    df_batters["plateTimeY"] - df_batters["szTop"]
)
df_batters["plateTimeY_BotDiff"] = (
    df_batters["plateTimeY"] - df_batters["szBottom"]
)

# Set challengesLeft based on inningHalf (0 = Away, 1 = Home)
df_batters["challengesLeft"] = np.where(
    df_batters["inningHalf"] == 0,
    df_batters["awayChallengesLeft"],
    df_batters["homeChallengesLeft"],
)

# Set netScore based on inningHalf (0 = Away, 1 = Home)
# Positive if batting team is winning, 0 if tied, negative if losing
df_batters["netScore"] = np.where(
    df_batters["inningHalf"] == 0,
    df_batters["prePlayAwayScore"]-df_batters["prePlayHomeScore"],
    df_batters["prePlayHomeScore"]-df_batters["prePlayAwayScore"],
)

# Add pitchTypeGroup feature (1=fastballs, 2=breaking balls, 3=other)
# Define the conditions based on pitchTypeId
conditions = [
    df_batters["pitchTypeId"].isin([5, 6, 7, 12]),  # Fastballs
    df_batters["pitchTypeId"].isin([2, 3, 10, 13, 14, 15]),  # Breaking Balls
]

# Define the corresponding values for each condition
choices = [1, 2]

# Apply the conditions; default to 3 (Other) if no condition is met
df_batters["pitchTypeGroup"] = np.select(conditions, choices, default=3)

# Create list of columns to use as features
feature_cols = [
    "inning",
    "prePitchBalls",
    "prePitchStrikes",
    "runnerOn1st",
    "runnerOn2nd",
    "runnerOn3rd",
    "outs",
    "plateTimeX",
    "plateTimeY_topDiff",
    "plateTimeY_BotDiff",
    "startSpeed",
    "challengesLeft",
    "netScore",
    "pitchTypeGroup",
]

# Create the features DataFrame
X = df_batters[feature_cols]
# Create classes
y = df_batters["hasABSChallenge"].map({1: "Challenged", 0: "Not Challenged"})

print(f"There are {len(X)} rows in the cleaned Features dataset.")

# View the features DataFrame
X.head(20)

In [ ]:
# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"There are {len(X_train)} rows in the training set and {len(X_test)} rows in the test set.")


In [ ]:
# =====================================================================
# Train and Evaluate Joint Model for predicting
# whether batter will challenge pitch
# =====================================================================

# Initialize and train the hybrid estimator
hybrid_nb = HybridNaiveBayes()
hybrid_nb.fit(X_train, y_train)

print("Class labels:", hybrid_nb.classes_)
print("Number of samples per class:", hybrid_nb.class_count_)
print("Prior probability of each class:", hybrid_nb.class_prior_)

# Predict outcomes using test set
y_pred = hybrid_nb.predict(X_test)

# Output final metrics for Naïve Bayes tab
print("--- Unified Hybrid Naïve Bayes Results ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
plt.title("Batter Challenge Confusion Matrix")
plt.ylabel("Actual Class")
plt.xlabel("Predicted Class")
plt.show()

## Retrain model with balanced prior_mode to account for class discrepency

In [ ]:
# =====================================================================
# Train and Evaluate Joint Model for predicting
# whether batter will challenge pitch
# with balanced prior_mode
# =====================================================================

# Initialize with balancing active and fine-tune with bias
hybrid_nb_balanced = HybridNaiveBayes(use_balancing=True, bias=1.0)
hybrid_nb_balanced.fit(X_train, y_train)

# Predict outcomes using test set
y_pred_balanced = hybrid_nb_balanced.predict(X_test)

print("--- Model 1: Calibrated Hybrid Naïve Bayes Results ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_balanced):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced))

sns.heatmap(confusion_matrix(y_test, y_pred_balanced), annot=True, fmt="d", cmap="Blues")
plt.title("Balanced Batter Challenge Confusion Matrix")
plt.ylabel("Actual Class")
plt.xlabel("Predicted Class")
plt.show()

# Using NB to predict whether a batter's challenge will be overturned

In [ ]:
# Removed pitch location data from feature list since those values
#  can explicitly determine if a pitch was a ball or strike
feature_cols2 = [
    "inning",
    "prePitchBalls",
    "prePitchStrikes",
    "runnerOn1st",
    "runnerOn2nd",
    "runnerOn3rd",
    "outs",
    "startSpeed",
    "challengesLeft",
    "netScore",
    "pitchTypeGroup",
]

# Create new dataset that only contains challenged pitches
df_batters_ChallengedPitches = df_batters[df_batters["hasABSChallenge"] == 1]

# Create the features DataFrame
X = df_batters_ChallengedPitches[feature_cols2]
# Create classes
y = df_batters_ChallengedPitches["isOverturned"].map({1: "Overturned", 0: "Confirmed"})

print(f"There are {len(X)} rows in the cleaned Features dataset.")

# View the features DataFrame
X.head(20)

In [ ]:
# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"There are {len(X_train)} rows in the training set and {len(X_test)} rows in the test set.")

In [ ]:
# =====================================================================
# Train and Evaluate Joint Model for predicting
# whether batter's challenge gets overturned
# =====================================================================

# Initialize and train the hybrid estimator
hybrid_nb = HybridNaiveBayes()
hybrid_nb.fit(X_train, y_train)

print("Class labels:", hybrid_nb.classes_)
print("Number of samples per class:", hybrid_nb.class_count_)
print("Prior probability of each class:", hybrid_nb.class_prior_)

# Predict outcomes using test set
y_pred = hybrid_nb.predict(X_test)

# Output final metrics for Naïve Bayes tab
print("--- Unified Hybrid Naïve Bayes Results ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
plt.title("Batter Challenge Outcome Confusion Matrix")
plt.ylabel("Actual Class")
plt.xlabel("Predicted Class")
plt.show()

# Using NB to predict whether catcher/pitcher will challenge

In [ ]:
# Load defensive data
df_def = pd.read_csv(defense_url, encoding="latin1")

# Remove rows with null pitch locations
df_def = df_def.dropna(subset=["plateTimeY", "plateTimeX"])

# Feature Engineering: Calculate vertical distance deviations from zone edges
# Positive values mean above the edge, negative values mean below the edge
df_def["plateTimeY_topDiff"] = (
    df_def["plateTimeY"] - df_def["szTop"]
)
df_def["plateTimeY_BotDiff"] = (
    df_def["plateTimeY"] - df_def["szBottom"]
)

# Set challengesLeft based on inningHalf
df_def["challengesLeft"] = np.where(
    df_def["inningHalf"] == 1,
    df_def["awayChallengesLeft"],
    df_def["homeChallengesLeft"],
)

# Set netScore based on inningHalf
# Positive if defensive team is winning, 0 if tied, negative if losing
df_def["netScore"] = np.where(
    df_def["inningHalf"] == 1,
    df_def["prePlayAwayScore"]-df_def["prePlayHomeScore"],
    df_def["prePlayHomeScore"]-df_def["prePlayAwayScore"],
)

# Add pitchTypeGroup feature (1=fastballs, 2=breaking balls, 3=other)
# Define the conditions based on pitchTypeId
conditions = [
    df_def["pitchTypeId"].isin([5, 6, 7, 12]),  # Fastballs
    df_def["pitchTypeId"].isin([2, 3, 10, 13, 14, 15]),  # Breaking Balls
]

# Define the corresponding values for each condition
choices = [1, 2]

# Apply the conditions; default to 3 (Other) if no condition is met
df_def["pitchTypeGroup"] = np.select(conditions, choices, default=3)

# Create list of columns to use as features
feature_cols = [
    "inning",
    "prePitchBalls",
    "prePitchStrikes",
    "runnerOn1st",
    "runnerOn2nd",
    "runnerOn3rd",
    "outs",
    "plateTimeX",
    "plateTimeY_topDiff",
    "plateTimeY_BotDiff",
    "startSpeed",
    "challengesLeft",
    "netScore",
    "pitchTypeGroup",
]

# Create the features DataFrame
X = df_def[feature_cols]
# Create classes
y = df_def["hasABSChallenge"].map({1: "Challenged", 0: "Not Challenged"})

print(f"There are {len(X)} rows in the cleaned Features dataset.")

# View the features DataFrame
X.head(20)

In [ ]:
# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"There are {len(X_train)} rows in the training set and {len(X_test)} rows in the test set.")

In [ ]:
# =====================================================================
# Train and Evaluate Joint Model for predicting
# whether pitcher/catcher will challenge pitch
# =====================================================================

# Initialize and train the hybrid estimator
hybrid_nb = HybridNaiveBayes()
hybrid_nb.fit(X_train, y_train)

print("Class labels:", hybrid_nb.classes_)
print("Number of samples per class:", hybrid_nb.class_count_)
print("Prior probability of each class:", hybrid_nb.class_prior_)

# Predict outcomes using test set
y_pred = hybrid_nb.predict(X_test)

# Output final metrics for Naïve Bayes tab
print("--- Unified Hybrid Naïve Bayes Results ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
plt.title("Defense Challenge Confusion Matrix")
plt.ylabel("Actual Class")
plt.xlabel("Predicted Class")
plt.show()

## Retrain model with balanced prior_mode to account for class discrepency

In [ ]:
# =====================================================================
# Train and Evaluate Joint Model for predicting
# whether pitcher/catcher will challenge pitch
# with balanced prior_mode
# =====================================================================

# Initialize with balancing active and fine-tune with bias
hybrid_nb_balanced = HybridNaiveBayes(use_balancing=True, bias=1.0)
hybrid_nb_balanced.fit(X_train, y_train)

# Predict outcomes using test set
y_pred_balanced = hybrid_nb_balanced.predict(X_test)

print("--- Model 1: Calibrated Hybrid Naïve Bayes Results ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred_balanced):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_balanced))
print("\nClassification Report:")
print(classification_report(y_test, y_pred_balanced))

sns.heatmap(confusion_matrix(y_test, y_pred_balanced), annot=True, fmt="d", cmap="Blues")
plt.title("Balanced Defense Challenge Confusion Matrix")
plt.ylabel("Actual Class")
plt.xlabel("Predicted Class")
plt.show()

# Using NB to predict whether a catcher/pitcher's challenge will be overturned

In [ ]:
# Removed pitch location data from feature list since those values
#  can explicitly determine if a pitch was a ball or strike
feature_cols2 = [
    "inning",
    "prePitchBalls",
    "prePitchStrikes",
    "runnerOn1st",
    "runnerOn2nd",
    "runnerOn3rd",
    "outs",
    "startSpeed",
    "challengesLeft",
    "netScore",
    "pitchTypeGroup",
]

# Create new dataset that only contains challenged pitches
df_def_ChallengedPitches = df_def[df_def["hasABSChallenge"] == 1]

# Create the features DataFrame
X = df_def_ChallengedPitches[feature_cols2]
# Create classes
y = df_def_ChallengedPitches["isOverturned"].map({1: "Overturned", 0: "Confirmed"})

print(f"There are {len(X)} rows in the cleaned Features dataset.")

# View the features DataFrame
X.head(20)

In [ ]:
# Split the dataset into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
print(f"There are {len(X_train)} rows in the training set and {len(X_test)} rows in the test set.")

In [ ]:
# =====================================================================
# Train and Evaluate Joint Model for predicting
# whether pitcher/catcher's challenge gets overturned
# =====================================================================

# Initialize and train the hybrid estimator
hybrid_nb = HybridNaiveBayes()
hybrid_nb.fit(X_train, y_train)

print("Class labels:", hybrid_nb.classes_)
print("Number of samples per class:", hybrid_nb.class_count_)
print("Prior probability of each class:", hybrid_nb.class_prior_)

# Predict outcomes using test set
y_pred = hybrid_nb.predict(X_test)

# Output final metrics for Naïve Bayes tab
print("--- Unified Hybrid Naïve Bayes Results ---")
print(f"Accuracy Score: {accuracy_score(y_test, y_pred):.4f}\n")
print("Confusion Matrix:")
print(confusion_matrix(y_test, y_pred))
print("\nClassification Report:")
print(classification_report(y_test, y_pred))

sns.heatmap(confusion_matrix(y_test, y_pred), annot=True, fmt="d", cmap="Blues")
plt.title("Defense Challenge Outcome Confusion Matrix")
plt.ylabel("Actual Class")
plt.xlabel("Predicted Class")
plt.show()